In [ ]:
import molpot as mpot
import molexp as me
from pathlib import Path
from burr.core import action, State, Action
from burr.core.parallelism import MapStates
from typing import Generator
import burr
from typing import Any
from burr.lifecycle.base import PreRunStepHook

In [ ]:
config = {
    "molecules": me.ParamSpace(['aspirin', 'toluene']),
    "split": {"train": 0.95, "eval": 0.05},
    "instances": 3,
    "steps": 1e6,
    'batch_size': 10
}

In [ ]:
from burr.core import action
import torch
from ignite.metrics import MeanAbsoluteError, BatchWise
from pathlib import Path

@action(reads=[], writes=['single_ds'])
def prepare_data(state: State) -> State:
    ds = mpot.dataset.rMD17(
        molecule=state['molecule'],
        save_dir=Path(state['work_dir']/"data"),
        device='cuda',
    )
    ds.add_process(mpot.process.NeighborList(cutoff=5.0))
    return state.update(**{f"single_ds": "ds"})

@action(reads=['single_ds'], writes=['single_dl'])
def prepare_dataloader(state: State) -> State:
    if "split" in state:
        ds_list = torch.utils.data.random_split(
            state['single_ds'],
            list(state['split'].values()),
        )
        ds_dict = dict(zip(state['split'].keys(), ds_list))
    else:
        ds_list = [state['single_ds']]
        ds_dict = {'train': ds_list[0]}
    dl_dict = {
        key: mpot.DataLoader(
            ds,
            batch_size=state['batch_size'],
            shuffle=True,
        ) for key, ds in ds_dict.items()
    }
    return state.update(**dl_dict)

@action(reads=['single_dl'], writes=['train_result'])
def train_model(state: State) -> State:

    eval_dl = state['single_dl']['eval']
    train_dl = state['single_dl']['train']

    pinet = mpot.potential.nnp.PiNet(
        depth=5,
        basis_fn=mpot.potential.nnp.radial.GaussianRBF(10, 5.0),
        cutoff_fn=mpot.potential.nnp.cutoff.CosineCutoff(5.0),
        pi_nodes=[64, 64],
        ii_nodes=[64, 64, 64, 64],
        pp_nodes=[64, 64, 64, 64],
        activation=torch.nn.Tanh(),
        rank=1
    )
    e_readout = mpot.potential.nnp.readout.Atomwise(
        [64, 1],
        from_key=("pinet", "p1"),
        to_key=("predicts", "energy")
    )
    f_readout = mpot.potential.nnp.readout.DPairPot(
        fx_key=("predicts", "energy"),
        dx_key=("pairs", "dist"),
        to_key=("predicts", "forces"),
        create_graph=True
    )
    model = mpot.potential.PotentialSeq("pinet", pinet, e_readout, f_readout)

    loss_fn = mpot.loss.MultiTargetLoss(torch.nn.MSELoss(), [("forces", "forces", 1.0)])

    trainer = mpot.PotentialTrainer(
        model,
        optimizer=torch.optim.Adam(model.parameters(), lr=1e-4),
        loss_fn=loss_fn,
        device="cuda",
    )
    trainer.add_evaluator(eval_dl, no_grad=False)

    def get_energy(data):
        return (data[0]["energy"], data[1]["energy"])

    def get_forces(data):
        return (data[0]["forces"], data[1]["forces"])

    trainer.add_metric(
        "e_mae",
        MeanAbsoluteError(output_transform=get_energy),
        target="all",
        usage=BatchWise(),
    )
    trainer.add_metric(
        "f_mae",
        MeanAbsoluteError(output_transform=get_forces),
        target="all",
        usage=BatchWise(),
    )

    trainer.attach_progressbar()
    from ignite.handlers import global_step_from_engine
    from ignite.handlers.tensorboard_logger import TensorboardLogger
    from ignite.engine.events import Events
    tb_logger = TensorboardLogger(Path("rmd17_log"))
    tb_logger.attach_output_handler(
        trainer.trainer,
        event_name=Events.ITERATION_COMPLETED,
        tag="trainer",
        output_transform=lambda x: {"loss": x[-1]},
        global_step_transform=global_step_from_engine(trainer.trainer),
    )

    tb_logger.attach_output_handler(
        trainer.trainer,
        event_name=Events.ITERATION_COMPLETED,
        tag="trainer",
        metric_names=["e_mae", "f_mae"],
        global_step_transform=global_step_from_engine(trainer.trainer),
    )

    tb_logger.attach_output_handler(
        trainer.evaluator,
        event_name=Events.EPOCH_COMPLETED,
        tag="evaluator",
        metric_names=["e_mae", "f_mae"],
        global_step_transform=global_step_from_engine(trainer.trainer),
    )

    _state = trainer.run(train_dl, max_epochs=1000)
    state = state.update(train_result=_state)

In [ ]:
class PrepareData(MapStates):

    def action(self, state, inputs) -> Action:
        return prepare_data
    
    def states(self, state: State, context, inputs) -> Generator[State, None, None]:
        for tag in inputs['molecules']:
            yield state.update(molecule=tag)

    def reduce(self, state: State, results: Generator[State, None, None]) -> State:
        all_ds = []
        for sub_state in results:
            all_ds.append(sub_state["single_ds"])
        return state.update(rmd17_ds=all_ds)
    
    @property
    def inputs(self) -> list[str]:
        return ['molecules'] + super().inputs
    
    @property
    def reads(self) -> list[str]:
        return []
    
    @property
    def writes(self) -> list[str]:
        return ['rmd17_ds']

In [ ]:
app = me.Project("rmd17", path="./rmd17").with_actions(
    prepare_data=PrepareData(),
    prepare_dataloader=prepare_dataloader,
    train_model=train_model
).with_transitions(("prepare_data", "prepare_dataloader"), ('prepare_dataloader', "train_model")).with_entrypoint("prepare_data").build()
app.run(inputs=config)